
# Intro to Python



### Start a small Python session

Fabric pure Python notebooks run on VM-backed compute.

For this demo, I’m requesting a small 2 vCore session so the notebook starts quickly and we can focus on the workflow instead of waiting on compute.

For larger datasets or heavier Python workloads, you can choose more vCores.

> Python notebooks scale up by choosing a VM size.  
> PySpark notebooks scale out by using Spark compute.


In [ ]:

%%configure -f
{
    "vCores": 2
}



## Let's start with the mandatory 'Hello World' – because tradition matters!


In [ ]:

print('Hello World!')



## Libraries



### Install library
- ### Example Pandas, Polars, DuckDB 


In [ ]:

%pip install matplotlib



### Install if not installed, Upgrade if installed and Upgrade dependencies


In [ ]:

%pip install -U matplotlib



### Is Library installed?



#### Specific library with detail


In [ ]:

%pip show duckdb



#### List installed libraries


In [ ]:

%pip list



#### List and filter using grep


In [ ]:

%pip list | grep duck



#### List that we can display and filter


In [ ]:

# Method that gives us more options like displaying and filtering
# uses newer method and not ones that are deprecated

from importlib.metadata import distributions
import pandas as pd

packages = [(dist.metadata['Name'], dist.version) for dist in distributions()]
# Give human readable column names
df = pd.DataFrame(packages, columns=['Package', 'Version'])
display(df.sort_values('Package'))



### Import library - alias friendly shorter names (preference)


In [ ]:

import matplotlib.pyplot as plt
import numpy as np


In [ ]:

x = np.linspace(0, 2 * np.pi, 200)
y = np.sin(x)

fig, ax = plt.subplots()
ax.plot(x, y)
plt.show()



## Python is Object Orientated



### Variables


In [ ]:

# Code for Variables

my_variable = 100
print(my_variable)


In [ ]:

hello = "Hello World!"
print(hello)



### f-strings — the clean way to build strings
Use `f"..."` and put variable names inside `{}` — Python fills them in at runtime.
You'll see this pattern constantly in notebook code.


In [ ]:

airline = "American Airlines"
origin = "DCA"
distance = 2288

# Old way
print("Airline: " + airline + ", From: " + origin + ", Distance: " + format(distance, ",") + " miles")

# f-string way
print(f"Airline: {airline}, From: {origin}, Distance: {distance:,} miles")



### Functions


In [ ]:

# Code for Functions
# We have already seen one: print()
# We apply this over the object
# Here is another

fruit = 'apple'
fruit_length = len(fruit)
print(fruit_length)


In [ ]:

# We can create our own

def double(x):
    return x * 2

print(double(10))



### Methods - part of the object


In [ ]:

# Code for Methods

name = "DAX Shepherd"
lowername = name.lower()    # Method to convert to lowercase
lowername.startswith("d")        



### Method Chaining


In [ ]:

# Chaining

name = "DAX Shepherd"

#First convert to lowercase and then check if it start with lowercase "d"

name.lower().startswith("d")



### Chaining across lines



### **Method 1: Parenthesis '()'**


In [ ]:

# Chaining across lines

# Very common in PySpark code
# Allows you to think about each transformtion you are doing

# Use the parenthesis '()'

name = "DAX Shepherd"

result = (
    name
    .lower()
    .startswith("d")
)

result



### **Method 2: Backslash '\\'**


In [ ]:

# Chaining across lines

# Very common in PySpark code
# Allows you to think about each transformation you are doing

# Use the backslash '\'

name = "DAX Shepherd"

name \
    .lower() \
    .startswith("d")



### Accessing data from Pure Python notebook



- #### Drag File over — DuckDB reads the 650 MB CSV directly


In [ ]:

import duckdb

flights_path = "/lakehouse/default/Files/flights/flights_2022.csv"
display(duckdb.sql(f"SELECT * FROM read_csv('{flights_path}') LIMIT 1000").df())



- #### Small lookup files — pandas is perfect here


In [ ]:

import pandas as pd
airlines = pd.read_csv("/lakehouse/default/Files/flights/airlines.csv")
display(airlines)



### Using DuckDB Example - adding our own SQL Code


In [ ]:

import duckdb

duckdb.sql(
    """
    SELECT
        Origin,
        Reporting_Airline,
        COUNT(*) AS Cancelled_Flights
    FROM read_csv('/lakehouse/default/Files/flights/flights_2022.csv')
    WHERE Cancelled = 1
        AND Origin IN ('DCA', 'IAD', 'BWI')
    GROUP BY Origin, Reporting_Airline
    ORDER BY Origin, Cancelled_Flights DESC
    LIMIT 15
    """
).df()



## Pandas - Working with Data
Pandas is the go-to library for data work in Python.
Everything here has a direct equivalent in PySpark — same ideas, different syntax.



### Inspect the data


In [ ]:

import pandas as pd

airports = pd.read_csv("/lakehouse/default/Files/flights/airports.csv")

print(airports.shape)      # rows, columns
print(airports.dtypes)     # column types
airports.head(5)



### Filter rows
We're presenting at Power BI Days DC — let's look at the DC metro airports


In [ ]:

dc_airports = airports[airports["iata_code"].isin(["DCA", "IAD", "BWI"])]
display(dc_airports)



### Select columns
Only the columns we care about


In [ ]:

slim = airports[["iata_code", "name", "municipality", "iso_region", "type"]]
display(slim.head(10))



### GroupBy + Aggregation
Count airports by type — same concept as PySpark's `.groupBy().agg()`


In [ ]:

airports_by_type = (
    airports
    .groupby("type")
    .agg(airport_count=("iata_code", "count"))
    .reset_index()
    .sort_values("airport_count", ascending=False)
)

display(airports_by_type)



### Merge (Join)
Add the departure airport name to a sample of flights — same concept as PySpark's `.join()`


In [ ]:

flights_sample = pd.read_csv("/lakehouse/default/Files/flights/flights_2022.csv", nrows=50000)

dc_flights = flights_sample[flights_sample["Origin"].isin(["DCA", "IAD", "BWI"])]

dc_with_airport = (
    dc_flights
    .merge(airports[["iata_code", "name", "municipality"]], left_on="Origin", right_on="iata_code", how="left")
    .rename(columns={"name": "Origin Airport", "municipality": "Origin City"})
    .drop(columns=["iata_code"])
)

display(dc_with_airport[["FlightDate", "Reporting_Airline", "Origin", "Origin Airport", "Origin City", "Dest"]].head(10))



# What happens when your data is too big??


In [ ]:

from IPython.display import Image, display
display(Image(filename="/lakehouse/default/Files/demo_assets/Out_Of_Memory.jpg", width=1000))
display(Image(filename="/lakehouse/default/Files/demo_assets/this_is_fine.jpg", width=800))
